## 1. Imports and Dependencies

In [ ]:
import json
import faiss
import numpy as np
import requests

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

## 2. Load Structured MNE Documentation Corpus

In [5]:
with open("mne_docs_test.json", "r") as f:
    documents = json.load(f)

print(f"Loaded {len(documents)} documentation chunks.")

Loaded 60 documentation chunks.


## 3. Load Embeddings and FAISS Index

In [7]:
embeddings = np.load("mne_embeddings.npy")

index = faiss.read_index("mne_faiss.index")

print("Embeddings shape:", embeddings.shape)
print("FAISS index loaded successfully.")

Embeddings shape: (60, 384)
FAISS index loaded successfully.


## 4. Load Embedding Model

In [8]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


## 5. Generate Query Embedding

In [9]:
query = """
Generate constraints and test cases
for mne.io.read_raw_edf
"""

query_embedding = embedding_model.encode([query])

query_embedding = np.array(
    query_embedding,
    dtype="float32"
)

print("Query embedding shape:", query_embedding.shape)

Query embedding shape: (1, 384)


## 6. Retrieve Top-5 Relevant Documentation Chunks

In [10]:
k = 5

distances, indices = index.search(
    query_embedding,
    k
)

print("Retrieved indices:", indices)

Retrieved indices: [[ 0 52 55 34  5]]


## 7. Construct Retrieved Context

In [14]:
retrieved_context = ""

for rank, idx in enumerate(indices[0]):

    doc = documents[idx]

    retrieved_context += f"""
    
Rank: {rank + 1}

Function:
{doc["function_name"]}

Documentation:
{doc["description"]}

Parameters:
{json.dumps(doc["parameters"], indent=2)}

"""

print(retrieved_context[:5000])



Rank: 1

Function:
mne.io.read_raw_edf

Documentation:


Parameters:
{
  "input_fname path-like": "Path to the EDF or EDF+ file or EDF/EDF+ file itself. If a file-like\nobject is provided, preload must be used. Changed in version 1.10: Added support for file-like objects",
  "eog list or tuple": "Names of channels or list of indices that should be designated EOG\nchannels. Values should correspond to the electrodes in the file.\nDefault is None.",
  "misc list or tuple": "Names of channels or list of indices that should be designated MISC\nchannels. Values should correspond to the electrodes in the file.\nDefault is None.",
  "stim_channel 'auto' | str | list of str | int | list of int": "Defaults to 'auto' , which means that channels named 'status' or 'trigger' (case insensitive) are set to STIM. If str (or list of\nstr), all channels matching the name(s) are set to STIM. If int (or\nlist of ints), channels corresponding to the indices are set to STIM.",
  "exclude list of str | str

## 8. Construct Grounded Prompt

In [13]:
prompt = f"""
You are an API constraint and test generation system.

Using ONLY the retrieved API documentation below,
generate parameter-level constraints and corresponding test cases.

For each inferred constraint provide:

1. Parameter Name
2. Constraint
3. Short Reasoning
4. Valid Example
5. Invalid pytest-style Test Case

Focus on:
- datatype constraints
- invalid input conditions
- mutually conflicting parameters
- filesystem-related failures
- boundary conditions

Rules:
- ONLY use behaviors supported by the documentation
- Do NOT invent undocumented parameters
- Do NOT assume hidden implementation details
- Prefer parameter-level reasoning over generic testing
- Keep outputs concise and structured

Retrieved Documentation:
{retrieved_context}
"""



## 9. Generate Constraints Using Qwen3 via Ollama

In [15]:
url = "http://localhost:11434/api/generate"

payload = {
    "model": "qwen3:8b-q4_K_M",
    "prompt": prompt,
    "stream": False
}

response = requests.post(
    url,
    json=payload
)

result = response.json()

print(result["response"])

It seems you've shared a list of parameters and attributes from the **MNE-Python** library's `mne.io.Raw` class, which is used for handling raw EEG, MEG, and other neural data. Here's a breakdown of what this text represents and how to interpret it:

---

### **Key Concepts from the Text**
1. **`h_freq`**  
   - **Purpose**: Approximate high-pass cutoff frequency for filtering (e.g., to remove slow drifts).  
   - **Note**: Uses Savitzky-Golay filtering (polynomial smoothing) instead of traditional FIR/IIR filters.  

2. **`ref_channels`**  
   - **Purpose**: Specifies reference channels for re-referencing data (e.g., "average" for average reference, "REST" for reference electrode standardization).  
   - **Example**: `{'A1': ['A2', 'A3']}` replaces channel A1 with A1 - mean(A2, A3).  

3. **`montage`**  
   - **Purpose**: Assigns physical positions to channels (e.g., for EEG caps or MEG sensors).  
   - **Example**: `"easycap-M1"` uses a built-in montage.  

4. **`scalings`**  
   - *